## Part I: Confusion Matrix Evaluation

This section calculates precision, recall, and their macro- and micro-averages from a confusion matrix.

In [35]:
# Define class labels
classes = ["Cat", "Dog", "Rabbit"]

# Define the confusion matrix
# Rows are actual classes, columns are predicted classes.
confusion_matrix = [
    [5, 10, 5],  # Actual Cat: 5 correct, 10 as Dog, 5 as Rabbit
    [15, 20, 10], # Actual Dog: 15 as Cat, 20 correct, 10 as Rabbit
    [0, 15, 10]   # Actual Rabbit: 0 as Cat, 15 as Dog, 10 correct
]

# Number of classes
n = len(classes)

In [36]:
precisions = []
recalls = []

# Calculate per-class precision and recall
for i in range(n):
    tp = confusion_matrix[i][i] # True Positives (diagonal element)
    fp = sum(confusion_matrix[i]) - tp # False Positives (row sum - TP)
    fn = sum(confusion_matrix[j][i] for j in range(n)) - tp # False Negatives (column sum - TP)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    precisions.append(precision)
    recalls.append(recall)

    print(classes[i])
    print("TP =", tp)
    print("FP =", fp)
    print("FN =", fn)
    print("Precision =", round(precision, 3))
    print("Recall =", round(recall, 3))
    print()

Cat
TP = 5
FP = 15
FN = 15
Precision = 0.25
Recall = 0.25

Dog
TP = 20
FP = 25
FN = 25
Precision = 0.444
Recall = 0.444

Rabbit
TP = 10
FP = 15
FN = 15
Precision = 0.4
Recall = 0.4



In [37]:
# Macro-averaged Precision: average of per-class precisions
macro_precision = sum(precisions) / n
# Macro-averaged Recall: average of per-class recalls
macro_recall = sum(recalls) / n

print("Macro Precision =", round(macro_precision, 3))
print("Macro Recall =", round(macro_recall, 3))

Macro Precision = 0.365
Macro Recall = 0.365


## Part II: Bigram Language Model Implementation

This section builds a bigram language model to calculate unigram/bigram counts and probabilities, then uses these to score sentences.

In [38]:
total_tp = 0
total_fp = 0
total_fn = 0

# Sum TP, FP, FN across all classes
for i in range(n):
    tp = confusion_matrix[i][i] # True Positives
    fp = sum(confusion_matrix[i]) - tp # False Positives
    fn = sum(confusion_matrix[j][i] for j in range(n)) - tp # False Negatives

    total_tp += tp
    total_fp += fp
    total_fn += fn

# Micro-averaged Precision: Total TP / (Total TP + Total FP)
micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
# Micro-averaged Recall: Total TP / (Total TP + Total FN)
micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0

print("Micro Precision =", round(micro_precision, 3))
print("Micro Recall =", round(micro_recall, 3))

Micro Precision = 0.389
Micro Recall = 0.389


In [39]:
# Training corpus: list of sentences, each a list of words.
# '<s>' = start of sentence, '</s>' = end of sentence.
corpus = [
    ["<s>", "I", "love", "NLP", "</s>"],
    ["<s>", "I", "love", "deep", "learning", "</s>"],
    ["<s>", "deep", "learning", "is", "fun", "</s>"]
]

print(corpus)

[['<s>', 'I', 'love', 'NLP', '</s>'], ['<s>', 'I', 'love', 'deep', 'learning', '</s>'], ['<s>', 'deep', 'learning', 'is', 'fun', '</s>']]


In [40]:
unigram_counts = {}
bigram_counts = {}

# Iterate sentences to count unigrams and bigrams
for sentence in corpus:

    # Unigram counts (frequency of single words)
    for word in sentence:
        unigram_counts[word] = unigram_counts.get(word, 0) + 1

    # Bigram counts (frequency of word pairs)
    for i in range(len(sentence) - 1):
        bigram = (sentence[i], sentence[i + 1])
        bigram_counts[bigram] = bigram_counts.get(bigram, 0) + 1

print("Unigram Counts:")
print(unigram_counts)

print("\nBigram Counts:")
print(bigram_counts)

Unigram Counts:
{'<s>': 3, 'I': 2, 'love': 2, 'NLP': 1, '</s>': 3, 'deep': 2, 'learning': 2, 'is': 1, 'fun': 1}

Bigram Counts:
{('<s>', 'I'): 2, ('I', 'love'): 2, ('love', 'NLP'): 1, ('NLP', '</s>'): 1, ('love', 'deep'): 1, ('deep', 'learning'): 2, ('learning', '</s>'): 1, ('<s>', 'deep'): 1, ('learning', 'is'): 1, ('is', 'fun'): 1, ('fun', '</s>'): 1}


In [41]:
bigram_probabilities = {}

# Calculate bigram probabilities P(word2 | word1)
for bigram in bigram_counts:

    word1 = bigram[0]
    word2 = bigram[1]

    # MLE: Count(word1, word2) / Count(word1)
    probability = bigram_counts[bigram] / unigram_counts[word1]

    bigram_probabilities[bigram] = probability

print("Bigram Probabilities:")

# Display all calculated bigram probabilities
for bigram in bigram_probabilities:
    print(bigram, "=", bigram_probabilities[bigram])

Bigram Probabilities:
('<s>', 'I') = 0.6666666666666666
('I', 'love') = 1.0
('love', 'NLP') = 0.5
('NLP', '</s>') = 1.0
('love', 'deep') = 0.5
('deep', 'learning') = 1.0
('learning', '</s>') = 0.5
('<s>', 'deep') = 0.3333333333333333
('learning', 'is') = 0.5
('is', 'fun') = 1.0
('fun', '</s>') = 1.0


In [42]:
def sentence_probability(sentence):

    probability = 1

    # Calculate sentence probability by multiplying bigram probabilities
    for i in range(len(sentence) - 1):
        bigram = (sentence[i], sentence[i + 1])

        # If bigram exists, multiply its probability
        if bigram in bigram_probabilities:
            probability = probability * bigram_probabilities[bigram]
        else:
            # If bigram is new, sentence probability is 0
            return 0

    return probability

In [43]:
# Test sentences
sentence1 = ["<s>", "I", "love", "NLP", "</s>"]
sentence2 = ["<s>", "I", "love", "deep", "learning", "</s>"]

# Calculate probabilities
prob1 = sentence_probability(sentence1)
prob2 = sentence_probability(sentence2)

print("Sentence 1 probability =", prob1)
print("Sentence 2 probability =", prob2)

Sentence 1 probability = 0.3333333333333333
Sentence 2 probability = 0.16666666666666666


In [44]:
# Determine which sentence the model prefers
if prob1 > prob2:
    print("Model prefers Sentence 1")
    print("Reason: Sentence 1 has the higher probability.")
elif prob2 > prob1:
    print("Model prefers Sentence 2")
    print("Reason: Sentence 2 has the higher probability.")
else:
    print("Both sentences have the same probability.")

Model prefers Sentence 1
Reason: Sentence 1 has the higher probability.
